# PipelineTS All Models Guide
# PipelineTS 全部模型使用指南

This tutorial provides a detailed guide for all available models in PipelineTS, including:
本教程详细介绍 PipelineTS 中所有可用模型的使用方法，包括：

- **15 Neural Network Models / 15 个神经网络模型** (NN Models)
- **4 Machine Learning Models / 4 个机器学习模型** (ML Models)
- **2 Statistical Models / 2 个统计模型** (Statistic Models)
- **3 Foundation Models / 3 个基础模型** (Chronos-2 family, optional dependency / Chronos-2 家族，可选依赖)

All models follow the same unified `fit()` / `predict()` interface.
每个模型都遵循统一的 `fit()` / `predict()` 接口。

In [ ]:
import numpy as np
import pandas as pd

# Prepare example data / 准备示例数据
np.random.seed(42)
n = 200
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
values = np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.randn(n) * 0.1
data = pd.DataFrame({'date': dates, 'value': values})

LAGS = 12
PREDICT_N = 10

print(f"Data shape / 数据形状: {data.shape}")
data.head()

## Part I: Neural Network Models (NN Models)
## 第一部分：神经网络模型

All NN models share the following common parameters:
所有 NN 模型共享以下通用参数：

- `time_col`: Time column name / 时间列名
- `target_col`: Target column name / 目标列名
- `lags`: Number of past time steps (input window size) / 滞后步数（输入窗口大小）
- `quantile`: Coverage level for prediction intervals (set to None for point prediction only) / 预测区间分位数（设为 None 则只做点预测）
- `epochs`: Maximum training epochs / 训练轮数
- `patience`: Early stopping patience value / 早停耐心值
- `verbose`: Whether to display training information / 是否显示训练信息

### 1.1 NLinearModel
Simple linear mapping model, the fastest NN model.
简单线性映射模型，速度最快。

In [ ]:
from PipelineTS.nn_model import NLinearModel

model = NLinearModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.2 DLinearModel
Decomposition linear model that separates trend and seasonal components.
分解线性模型，将序列分解为趋势和季节性分量。

In [ ]:
from PipelineTS.nn_model import DLinearModel

model = DLinearModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.3 NBeatsModel
N-BEATS architecture supporting both generic and interpretable modes.
N-BEATS 模型，支持 generic 和 interpretable 两种架构。

In [ ]:
from PipelineTS.nn_model import NBeatsModel

# Generic 架构
model = NBeatsModel(
    time_col='date', target_col='value', lags=LAGS,
    generic_architecture=True, num_stacks=2, num_blocks=1,
    num_layers=2, layer_widths=64,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.4 NHitsModel
N-HiTS model with hierarchical interpolation for efficient forecasting.
N-HiTS 模型，采用分层插值结构提高预测效率。

In [ ]:
from PipelineTS.nn_model import NHitsModel

model = NHitsModel(
    time_col='date', target_col='value', lags=LAGS,
    num_stacks=2, num_blocks=1, num_layers=2, layer_widths=64,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.5 TFTModel
Temporal Fusion Transformer, combining LSTM and attention mechanisms.
时序融合 Transformer，结合 LSTM 和注意力机制。

In [ ]:
from PipelineTS.nn_model import TFTModel

model = TFTModel(
    time_col='date', target_col='value', lags=LAGS,
    hidden_size=32, lstm_layers=1, n_heads=2,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.6 TransformerModel
Classic Transformer encoder architecture for time series.
经典 Transformer 编码器架构。

In [ ]:
from PipelineTS.nn_model import TransformerModel

model = TransformerModel(
    time_col='date', target_col='value', lags=LAGS,
    d_model=32, nhead=2, num_encoder_layers=2, dim_feedforward=64,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.7 TiDEModel
Time-series Dense Encoder, a fully-connected encoder-decoder architecture.
时序密集编码器，基于全连接的编解码器结构。

In [ ]:
from PipelineTS.nn_model import TiDEModel

model = TiDEModel(
    time_col='date', target_col='value', lags=LAGS,
    num_encoder_layers=2, num_decoder_layers=2,
    hidden_size=64, decoder_output_dim=16,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.8 GAUModel
Gated Attention Unit model with gated attention mechanism.
门控注意力单元模型，使用门控注意力机制。

In [ ]:
from PipelineTS.nn_model import GAUModel

model = GAUModel(
    time_col='date', target_col='value', lags=LAGS,
    level=3, quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.9 StackingRNNModel
RWKV (linear RNN) encoder with gated residual blocks and RevIN normalization. Uses parallel linear temporal mixing (no sequential recurrence), followed by gated residual refinement with SiLU activation.
RWKV（线性 RNN）编码器 + 门控残差块 + RevIN 归一化。使用并行线性时序混合（无顺序递归），经过带 SiLU 激活的门控残差精炼。

In [ ]:
from PipelineTS.nn_model import StackingRNNModel

model = StackingRNNModel(
    time_col='date', target_col='value', lags=LAGS,
    blocks=2, quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.10 Time2VecModel
Trend-seasonal decomposition + StableTime2Vec periodic encoding + RWKV temporal mixing. The input is decomposed via moving average into trend and seasonal components; the seasonal path uses log-spaced Time2Vec features followed by RWKV encoder blocks. Includes RevIN normalization.
趋势-季节分解 + StableTime2Vec 周期编码 + RWKV 时序混合。输入通过移动平均分解为趋势和季节分量；季节路径使用对数间距 Time2Vec 特征 + RWKV 编码器。包含 RevIN 归一化。

In [ ]:
from PipelineTS.nn_model import Time2VecModel

model = Time2VecModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.11 PatchRNNModel
Patch-based RNN that segments the input sequence into patches before feeding to LSTM.
Patch + RNN 模型，将序列分块后输入 LSTM。

In [ ]:
from PipelineTS.nn_model import PatchRNNModel

model = PatchRNNModel(
    time_col='date', target_col='value', lags=LAGS,
    kernel_size=4, quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.12 TCNModel
Temporal Convolutional Network with dilated causal convolutions.
时序卷积网络，使用膨胀因果卷积。

In [ ]:
from PipelineTS.nn_model import TCNModel

model = TCNModel(
    time_col='date', target_col='value', lags=LAGS,
    kernel_size=3, quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.13 ITransformerModel
Inverted Transformer that treats each variable as a token. Supports multivariate prediction.
反转 Transformer，将每个变量视为一个 token，支持多变量预测。

In [ ]:
from PipelineTS.nn_model import ITransformerModel

model = ITransformerModel(
    time_col='date', target_col='value', lags=LAGS,
    d_model=32, n_heads=2, d_ff=64, e_layers=1,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.14 SRSNetModel
Selective Representation Space Network with multi-scale adaptive patches. Supports multivariate prediction.
选择性表征空间网络，多尺度自适应 patch + 选择性表征，支持多变量预测。

In [ ]:
from PipelineTS.nn_model import SRSNetModel

model = SRSNetModel(
    time_col='date', target_col='value', lags=LAGS,
    d_model=32, n_heads=2,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 1.15 DeepARModel
DeepAR autoregressive probabilistic forecasting model. Uses LSTM-based encoder-decoder with probabilistic output.
DeepAR 自回归概率预测模型。使用基于 LSTM 的编解码器和概率输出。

In [ ]:
from PipelineTS.nn_model import DeepARModel

model = DeepARModel(
    time_col='date', target_col='value', lags=LAGS,
    hidden_size=32, n_layers=2,
    quantile=0.9, epochs=50, patience=10, verbose=False
)
model.fit(data)
result = model.predict(PREDICT_N)
result

## Part II: Machine Learning Models (ML Models)
## 第二部分：机器学习模型

ML models are based on ensemble methods. They typically train faster than NN models.
ML 模型基于集成学习方法，训练速度通常更快。

All ML models automatically build rich lag features (26+ features per window).
所有 ML 模型自动构建丰富的滞后特征（每个窗口 26+ 个特征）。

### 2.1 WideGBRTModel
Wide-table GBRT model with automatically constructed rich time series features. Supports differencing.
宽表 GBRT 模型，自动构建丰富的时序特征。支持差分操作。

In [ ]:
from PipelineTS.ml_model import WideGBRTModel

model = WideGBRTModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9, n_estimators=200, verbose=-1,
    differential_n=1
)
model.fit(data)
result = model.predict(PREDICT_N)
result

### 2.2 MultiOutputRegressorModel / MultiStepRegressorModel / RegressorChainModel
Multi-output regression models for multi-step forecasting.
多输出回归模型，适用于多步预测。

In [ ]:
from PipelineTS.ml_model import (
    MultiOutputRegressorModel,
    MultiStepRegressorModel,
    RegressorChainModel
)

for ModelClass in [MultiOutputRegressorModel, MultiStepRegressorModel, RegressorChainModel]:
    model = ModelClass(
        time_col='date', target_col='value', lags=LAGS,
        quantile=0.9, verbose=-1
    )
    model.fit(data)
    result = model.predict(PREDICT_N)
    print(f"{ModelClass.__name__}: 预测 {len(result)} 步")

# TODO: Delete this cell (old GBDT model section removed)

In [ ]:
# TODO: Delete this cell (old GBDT model code removed)

# TODO: Delete this cell (old ML model section removed)

In [ ]:
# TODO: Delete this cell (old ML model code removed)

# TODO: Delete this cell (was duplicate WideGBRTModel header, now removed)

In [ ]:
# TODO: Delete this cell (was duplicate WideGBRTModel code, now removed)

# TODO: Delete this cell (was duplicate MultiOutput header, now removed)

In [ ]:
# TODO: Delete this cell (was duplicate MultiOutput code, now removed)

## Part III: Statistical Models
## 第三部分：统计模型

### 3.1 ProphetModel
Custom Prophet-like decomposable model (not Facebook Prophet). Uses piecewise linear trend + Fourier seasonality + optional causal rolling lag features, solved via ridge regression (closed-form). 100x+ faster than Facebook Prophet.
自定义类 Prophet 可分解模型（非 Facebook Prophet）。使用分段线性趋势 + 傅里叶季节性 + 可选因果滚动滞后特征，通过岭回归（解析解）求解。比 Facebook Prophet 快 100 倍以上。

In [ ]:
from PipelineTS.statistic_model import ProphetModel

model = ProphetModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9, auto_seasonality=True
)
model.fit(data, cv=2)
result = model.predict(PREDICT_N)
result

### 3.2 AutoARIMAModel
Automatic search for the best ARIMA parameters.
自动搜索最佳 ARIMA 参数。

In [ ]:
from PipelineTS.statistic_model import AutoARIMAModel

model = AutoARIMAModel(
    time_col='date', target_col='value', lags=LAGS,
    start_p=0, max_p=3, start_q=0, max_q=3,
    seasonal=False, quantile=0.9
)
model.fit(data, cv=2)
result = model.predict(PREDICT_N)
result

## Part IV: Foundation Models (Optional)
## 第四部分：基础模型（可选依赖）

Foundation models are large pretrained models that perform **zero-shot forecasting** — no training on your data is needed. They use pretrained weights from massive time series corpora.

基础模型是大型预训练模型，执行**零样本预测** —— 无需在您的数据上训练。它们使用来自大规模时序语料库的预训练权重。

> **Note / 注意:** Requires `pip install chronos-forecasting` as an optional dependency.
> 需要安装可选依赖：`pip install chronos-forecasting`

Three Chronos-2 model classes are available:
提供三个 Chronos-2 模型类：

| Class / 类 | Pipeline Key / 管道键名 | HuggingFace Path | Size / 大小 |
|---|---|---|---|
| `Chronos2Model` | `chronos_2` | `amazon/chronos-2` | 120M |
| `Chronos2SynthModel` | `chronos_2_synth` | `autogluon/chronos-2-synth` | 120M |
| `Chronos2SmallModel` | `chronos_2_small` | `autogluon/chronos-2-small` | 28M |

`ChronosModel` is a backward-compatible alias for `Chronos2Model`.
`ChronosModel` 是 `Chronos2Model` 的向后兼容别名。

### 4.1 Chronos2SmallModel (Lightweight, fastest)

The smallest Chronos-2 variant (28M params). Best for quick experimentation.
最小的 Chronos-2 变体（28M 参数），适合快速实验。

In [ ]:
# Chronos-2 requires: pip install chronos-forecasting
# Chronos-2 需要安装：pip install chronos-forecasting
try:
    from PipelineTS.nn_model import Chronos2SmallModel

    # Chronos2SmallModel — lightweight 28M param variant
    # Chronos2SmallModel — 轻量级 28M 参数变体
    model = Chronos2SmallModel(
        time_col='date', target_col='value',
        quantile=0.9,
    )
    model.fit(data, cv=2)
    result = model.predict(PREDICT_N)
    print("Chronos2SmallModel prediction / Chronos2SmallModel 预测:")
    display(result)

except ImportError:
    print("chronos-forecasting not installed. Install with: pip install chronos-forecasting")
    print("chronos-forecasting 未安装。请运行：pip install chronos-forecasting")

### 4.2 Other Chronos-2 Variants / 其他 Chronos-2 变体

```python
from PipelineTS.nn_model import Chronos2Model, Chronos2SynthModel, Chronos2SmallModel

# Chronos2Model — full 120M param model (amazon/chronos-2)
model = Chronos2Model(time_col='date', target_col='value', quantile=0.9)

# Chronos2SynthModel — 120M, trained on synthetic data (autogluon/chronos-2-synth)
model = Chronos2SynthModel(time_col='date', target_col='value', quantile=0.9)

# Chronos2SmallModel — lightweight 28M (autogluon/chronos-2-small)
model = Chronos2SmallModel(time_col='date', target_col='value', quantile=0.9)
```

#### Using Chronos-2 in Pipeline / 在 Pipeline 中使用 Chronos-2

```python
from PipelineTS.pipeline import ModelPipeline

# Use any of the 3 models via pipeline key / 通过管道键名使用任一模型
pipe = ModelPipeline(
    time_col='date', target_col='value', lags=12,
    include_models=['chronos_2_small'],  # or 'chronos_2', 'chronos_2_synth'
)
pipe.fit(data)
pipe.predict(n=10)
```

#### Chronos-2 with Covariates / Chronos-2 协变量支持

```python
# All Chronos-2 models support covariates / 所有 Chronos-2 模型支持协变量
model = Chronos2Model(
    time_col='date', target_col='value', quantile=0.9,
)
model.all_configs['known_covariates'] = ['holiday']
model.fit(data_with_covariates, cv=2)
future = pd.DataFrame({'holiday': [0, 0, 1, 0, 0]})
model.predict(5, future_covariates=future)
```

#### Chronos-2 with Multi-Series / Chronos-2 多序列

```python
model = Chronos2SmallModel(
    time_col='date', target_col='value', quantile=0.9,
)
model.all_configs['id_col'] = 'store'
model.fit(panel_data, cv=2)
model.predict(5)  # Returns predictions for all series / 返回所有序列的预测
```